# 04 - Merge curated h5ad

Two merges, **status-aware**. Genes are unified on `gene_symbol_upper` with `join='outer'`. No normalization / batch correction / scVI here.

* **merge 1** - only `raw_or_filtered_count` datasets
* **merge 2** - all curated datasets, keeping `data_status` in obs

In [ ]:
import sys
from pathlib import Path

# locate the v2 project root (folder containing config/dataset_manifest.yaml)
ROOT = Path.cwd()
while not (ROOT / "config" / "dataset_manifest.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

import manifest_utils as mf
man = mf.load_manifest()
paths = mf.project_paths(ROOT)
mf.ensure_dirs(paths)

import anndata_utils as au
import pandas as pd

curated = au.load_h5ad_collection(paths['curated'])
{k: (a.n_obs, a.n_vars, str(a.obs['data_status'].iloc[0]) if 'data_status' in a.obs else '?')
 for k, a in curated.items()}

## merge 1 - raw_or_filtered_count only

In [ ]:
raw_like = {k: a for k, a in curated.items()
            if 'data_status' in a.obs.columns
            and (a.obs['data_status'].astype(str) == 'raw_or_filtered_count').all()}
print('included:', list(raw_like.keys()))
merged_raw = au.merge_adatas(raw_like, join='outer', gene_key='gene_symbol_upper')
merged_raw

In [ ]:
out1 = paths['merged'] / 'merged_raw_or_filtered_count_als_sma_mouse.h5ad'
au.save_h5ad(merged_raw, out1, overwrite=True)

## merge 2 - all curated, status-aware

In [ ]:
merged_all = au.merge_adatas(curated, join='outer', gene_key='gene_symbol_upper')
print('data_status values present:')
print(merged_all.obs['data_status'].value_counts())
merged_all

In [ ]:
out2 = paths['merged'] / 'merged_all_status_aware_als_sma_mouse.h5ad'
au.save_h5ad(merged_all, out2, overwrite=True)

## Cell-count tables after merge

In [ ]:
keys = ['source_accession', 'dataset_id', 'disease_status', 'treatment',
        'tissue', 'enrichment', 'data_status']
au.cell_count_table(merged_all, keys)

Next: **05_check_merged_h5ad.ipynb**